- Notebook for creating datasets that is not site-aware split, but has still same properties (9:1 imbalance in all folds, same counts in all folds)

In [ ]:
import os
import sys
import numpy as np

data_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if data_root not in sys.path:
    sys.path.insert(0, data_root)

from finetuning.data_processing.dataset import load_dataset_source
from data.utils.general import get_sld
from data.utils.fold_splitting import analyze_folds

In [ ]:
from dotenv import load_dotenv

loaded = load_dotenv(".env")
if not loaded:
    loaded = load_dotenv("../../../.env")
assert loaded is True

In [ ]:
public_ours_dataset_source = load_dataset_source("public_ours")
public_folded_dataset = public_ours_dataset_source.df
public_folded_dataset["sld"] = public_folded_dataset["url"].apply(get_sld)

In [18]:

def assign_stratified_folds(df, n_folds=5, random_state=42):
    """
    Assign new fold IDs (0-4) while maintaining class distribution.
    Each fold will be approximately equal size with the same class proportions.
    """
    np.random.seed(random_state)
    
    print("Original label distribution:")
    label_counts = df['label'].value_counts().sort_index()
    print(label_counts)
    print(f"Total samples: {len(df):,}")
    
    result_df = df.copy()
    fold_assignments = np.zeros(len(df), dtype=int)
    
    for label in sorted(df['label'].unique()):
        label_mask = df['label'] == label
        label_indices = df[label_mask].index.values
        
        np.random.shuffle(label_indices)
        
        for i, idx in enumerate(label_indices):
            fold_assignments[idx] = i % n_folds
    
    result_df['fold'] = fold_assignments    
    return result_df

resampled_df1 = assign_stratified_folds(public_folded_dataset)
# verify_distribution(resampled_df1)
display(analyze_folds(resampled_df1))


In [19]:
target_path = os.path.join(os.getenv(f"OURS_PUBLIC_DATASET_FOLDER"), 'unfolded_version.csv')
resampled_df1 = resampled_df1.drop(columns=["sld"])
resampled_df1.to_csv(target_path, index=False)